In [35]:
# imports

import json
from pathlib import Path
from openai import OpenAI

In [36]:
def json_dir_to_jsonl(json_dir: str, output_jsonl_path: str) -> int:
    """Glob a directory for all JSON files and write them as one JSON object per line to a JSONL file."""
    json_paths = sorted(Path(json_dir).rglob("*.json"))
    with open(output_jsonl_path, "w") as out:
        for path in json_paths:
            with open(path, "r") as f:
                obj = json.load(f)
            out.write(json.dumps(obj, ensure_ascii=False) + "\n")
    return len(json_paths)



In [37]:
n = json_dir_to_jsonl("gdpr/train", "gdpr_train.jsonl")
print(f"Wrote {n:,} records to gdpr_train.jsonl")

Wrote 55 records to gdpr_train.jsonl


In [38]:
n = json_dir_to_jsonl("gdpr/test", "gdpr_test.jsonl")
print(f"Wrote {n:,} records to gdpr_test.jsonl")

Wrote 10 records to gdpr_test.jsonl


In [39]:
openai = OpenAI()

In [41]:
with open("gdpr_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [42]:
train_file

FileObject(id='file-NonYo19StZcxC3dESGaUZw', bytes=201367, created_at=1773242881, filename='gdpr_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [14]:
with open("gdpr_test.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [15]:
validation_file

FileObject(id='file-FLG6y9UEjSLyQcdE8scpUU', bytes=36660, created_at=1773242659, filename='gdpr_test.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [16]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="gdpr"
)

FineTuningJob(id='ftjob-bebQOipjlectvMdWvhVFFqQD', created_at=1773242663, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-xtqrRuCzyd5EbnxtW0MxUEpD', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-QHCsvMJ78cCU5zr7JLx9fr', validation_file='file-FLG6y9UEjSLyQcdE8scpUU', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='gdpr', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [57]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-bebQOipjlectvMdWvhVFFqQD', created_at=1773242663, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=1773243116, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-xtqrRuCzyd5EbnxtW0MxUEpD', result_files=[], seed=42, status='running', trained_tokens=None, training_file='file-QHCsvMJ78cCU5zr7JLx9fr', validation_file='file-FLG6y9UEjSLyQcdE8scpUU', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='gdpr', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experim

In [56]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [61]:
job_id = "ftjob-Rv2gbPw8E2VVxUMK28FmSf8O"

In [62]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-Rv2gbPw8E2VVxUMK28FmSf8O', created_at=1773234730, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:personal:gdpr:DIE8Fn4p', finished_at=1773236599, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-xtqrRuCzyd5EbnxtW0MxUEpD', result_files=['file-TcN2CzFYYCSeGbNGuzrBeV'], seed=42, status='succeeded', trained_tokens=46018, training_file='file-AgrfeVsB27Sch6Gm4yTGJV', validation_file='file-Hm1CQnVEmoFumdDMb5bUNm', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='gdpr', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=N

In [63]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-EdMJ02D0tJmbt3oXiQi2D931', created_at=1773237329, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-dr8bKZYmObJVe3iynwv2rfns', created_at=1773237328, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-geNVCmcvWQxIxmqqXSdkxNyx', created_at=1773237328, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:personal:gdpr:DIE8Fn4p passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blocking'}

In [64]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [65]:
print(fine_tuned_model_name)

ft:gpt-4.1-nano-2025-04-14:personal:gdpr:DIE8Fn4p


In [66]:
# The prompt

def test_messages_for():
    message = f"Assess the following school canteen cashless payment system notice for GDPR compliance:\n\n\"The school uses a cashless payment system for canteen purchases. Pupils are assigned a unique ID linked to their canteen account. Parents top up accounts online. Purchase history is visible to parents via the parent portal. Data is retained for the current academic year plus 1 year. We use CaterApp as our canteen management provider.\""
    return [
        {"role": "system", "content": "You are a GDPR compliance analyst. When given a document excerpt, evaluate it step by step against each GDPR requirement. For each step, quote the relevant text if found, or state it is missing. End with a structured verdict listing all violations. Always follow this exact format:\n\nStep 1 — Lawful basis (Art. 6): ...\nStep 2 — Special category data (Art. 9): ...\nStep 3 — Purpose limitation (Art. 5): ...\nStep 4 — Data minimization (Art. 5): ...\nStep 5 — Retention period (Art. 5): ...\nStep 6 — Data subject rights (Art. 15-21): ...\nStep 7 — Third-party disclosure (Art. 28): ...\nStep 8 — International transfers (Art. 46): ...\nStep 9 — DPO appointment (Art. 37): ...\nStep 10 — Breach notification (Art. 33): ...\n\nVERDICT: [Compliant / Non-Compliant / Partially Compliant]\nVIOLATIONS: [list each article breached]\nRECOMMENDATIONS: [one line per fix needed]"},
        {"role": "user", "content": message},
    ]

In [67]:
test_messages_for()

[{'role': 'system',
  'content': 'You are a GDPR compliance analyst. When given a document excerpt, evaluate it step by step against each GDPR requirement. For each step, quote the relevant text if found, or state it is missing. End with a structured verdict listing all violations. Always follow this exact format:\n\nStep 1 — Lawful basis (Art. 6): ...\nStep 2 — Special category data (Art. 9): ...\nStep 3 — Purpose limitation (Art. 5): ...\nStep 4 — Data minimization (Art. 5): ...\nStep 5 — Retention period (Art. 5): ...\nStep 6 — Data subject rights (Art. 15-21): ...\nStep 7 — Third-party disclosure (Art. 28): ...\nStep 8 — International transfers (Art. 46): ...\nStep 9 — DPO appointment (Art. 37): ...\nStep 10 — Breach notification (Art. 33): ...\n\nVERDICT: [Compliant / Non-Compliant / Partially Compliant]\nVIOLATIONS: [list each article breached]\nRECOMMENDATIONS: [one line per fix needed]'},
 {'role': 'user',
  'content': 'Assess the following school canteen cashless payment syste

In [68]:
# The inference function


def gpt_4__1_nano_fine_tuned():
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(),
        max_tokens=1000
    )
    return response.choices[0].message.content

In [69]:
# print(f"Step 1 — Lawful basis (Art. 6): [PARTIAL] Contract (Art. 6(1)(b)) for payment processing and public task (Art. 6(1)(e)) for school administration are implied but not stated.\n\nStep 2 — Special category data (Art. 9): [PARTIAL] Food purchase history can reveal dietary requirements linked to religion or health (e.g., halal, gluten-free, diabetic meals). No acknowledgement or safeguard stated.\n\nStep 3 — Purpose limitation (Art. 5): [PASS] Canteen payment processing with parental visibility is a clear and specific purpose.\n\nStep 4 — Data minimization (Art. 5): [PASS] Unique ID and purchase history are proportionate for cashless canteen management.\n\nStep 5 — Retention period (Art. 5): [PASS] Current academic year plus 1 year is clearly stated.\n\nStep 6 — Data subject rights (Art. 15-21): [FAIL] No data subject or parental rights mentioned.\n\nStep 7 — Third-party disclosure (Art. 28): [PARTIAL] CaterApp named as provider but no DPA reference stated.\n\nStep 8 — International transfers (Art. 46): [FAIL] Not addressed. CaterApp infrastructure may involve non-EEA processing.\n\nStep 9 — DPO appointment (Art. 37): [FAIL] Not addressed. Schools processing pupil data are required to appoint a DPO as public authorities.\n\nStep 10 — Breach notification (Art. 33): [FAIL] Not addressed.\n\nVERDICT: Non-Compliant\nVIOLATIONS: Art. 6 (lawful basis not stated), Art. 9 (dietary inference risk not addressed), Art. 15-21 (no rights), Art. 28 (CaterApp DPA not referenced), Art. 46 (transfer safeguards absent), Art. 37 (school DPO required), Art. 33 (no breach notification)\nRECOMMENDATIONS:\n- State lawful basis for each processing activity\n- Acknowledge dietary data may reveal religious or health information and state safeguards\n- Add parental rights section including access, rectification, and erasure\n- Reference CaterApp DPA and confirm EU/UK data processing\n- Address international transfer safeguards if CaterApp processes outside UK/EEA\n- Disclose school DPO contact\n- Add breach notification commitment")
print(gpt_4__1_nano_fine_tuned())

Step 1 — Lawful basis (Art. 6): [PARTIAL] Absent. The system processes Personal Data under an administrative/contractual necessity (Art. 6(1)(b)), but it is not explicitly identified as the lawful basis.

Step 2 — Special category data (Art. 9): [PASS] Unlikely. Food allergies (not specified) are not considered special category data unless linked to disability.

Step 3 — Purpose limitation (Art. 5): [PASS] The purposes are crystal clear: canteen control and parental oversight via purchase history.

Step 4 — Data minimization (Art. 5): [PARTIAL] Unique IDs are standard practice. Parent and pupil accounts are justified. Dietary or disability info is not hinted at but may be relevant for full minimisation.

Step 5 — Retention period (Art. 5): [PARTIAL] Current academic year plus 1 year — appropriate for school canteen purposes.

Step 6 — Data subject rights (Art. 15-21): [PARTIAL] Pupil rights are assumed upheld. Parental rights to erasure or portability (e.g., download purchase history) 